# Breakeven analysis: how much can spreads move before you lose money?

**Prerequisites:** notebook **05** (pricing fundamentals) and familiarity with carry decomposition metrics (`carry_total`, `cs01`, `dv01`).

Breakeven analysis answers: **"Given my carry over horizon H, how much can parameter P move against me before the total return hits zero?"**

This notebook demonstrates:
1. Linear breakeven (first-order: `-carry / sensitivity`)
2. Comparing breakeven across different horizons
3. Breakeven for different target parameters (spread vs. yield)

## Imports

In [1]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.valuations import ValuationResult, price_instrument_with_metrics

print("Ready.")

Ready.


## Setup: 5-year corporate bond and market

We'll use a 5% fixed-rate bond discounted at 4% — a bond trading above par with positive carry.

In [2]:
AS_OF = "2025-01-15"

bond_json = json.dumps({
    "type": "bond",
    "spec": {
        "id": "CORP-5Y",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "rate": 0.05,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "stub": "None",
                "end_of_month": False,
                "payment_lag_days": 0
            }
        },
        "discount_curve_id": "USD-OIS",
        "call_put": None,
        "attributes": {"tags": [], "meta": {}},
        "pricing_overrides": {}
    }
})

# Market: upward-sloping OIS curve
base = date.fromisoformat(AS_OF)
ois = DiscountCurve(
    "USD-OIS", base,
    [(0.0, 1.0), (0.5, 0.980), (1.0, 0.960), (2.0, 0.920),
     (3.0, 0.880), (5.0, 0.800), (10.0, 0.650)],
    day_count="act_365f",
)
mc = MarketContext()
mc.insert(ois)
market_json = mc.to_json()

print(f"Bond: 5Y 5% fixed, discounted at ~4%")
print(f"As-of: {AS_OF}")

Bond: 5Y 5% fixed, discounted at ~4%
As-of: 2025-01-15


## 1. Linear breakeven spread (6-month horizon)

The linear breakeven answers: assuming I earn carry for 6 months, how many basis points can the z-spread widen before my total return is zero?

Formula: `breakeven = -(carry_total) / cs01`

The `pricing_options` parameter passes `theta_period` (carry horizon) and `breakeven_config` (target + mode) as a JSON string.

In [3]:
opts_6m = json.dumps({
    "theta_period": "6M",
    "breakeven_config": {"target": "z_spread", "mode": "linear"}
})

result_json = price_instrument_with_metrics(
    bond_json, market_json, AS_OF,
    model="discounting",
    metrics=["carry_total", "coupon_income", "pull_to_par", "roll_down",
             "funding_cost", "cs01", "breakeven"],
    pricing_options=opts_6m,
)

vr = ValuationResult.from_json(result_json)

carry = vr.get_metric("carry_total")
cs01 = vr.get_metric("cs01")
be = vr.get_metric("breakeven")

print(f"Bond PV: ${vr.price:,.2f} {vr.currency}")
print()
print("Carry decomposition (6M horizon):")
for m in ["coupon_income", "pull_to_par", "roll_down", "funding_cost", "carry_total"]:
    v = vr.get_metric(m)
    if v is not None:
        print(f"  {m:20s}: ${v:>12,.2f}")
print()
print(f"CS01 (per bp):          ${cs01:>12,.2f}")
print(f"Breakeven spread:       {be:>12.1f} bp")
print()
print(f"Interpretation: spreads can widen by ~{be:.0f} bp over 6 months")
print(f"before your carry is wiped out.")

Bond PV: $1,045,996.90 USD

Carry decomposition (6M horizon):
  coupon_income       : $   25,000.00
  pull_to_par         : $   18,195.52
  roll_down           : $  -22,533.60
  funding_cost        : $        0.00
  carry_total         : $   20,661.92

CS01 (per bp):          $     -385.31
Breakeven spread:               53.6 bp

Interpretation: spreads can widen by ~54 bp over 6 months
before your carry is wiped out.


## 2. Breakeven across different horizons

Longer horizons mean more carry accrues, so the breakeven widens. Let's compare 1M, 3M, 6M, and 1Y.

In [4]:
horizons = ["1M", "3M", "6M", "1Y"]
results = {}

for h in horizons:
    opts = json.dumps({
        "theta_period": h,
        "breakeven_config": {"target": "z_spread", "mode": "linear"}
    })
    out = price_instrument_with_metrics(
        bond_json, market_json, AS_OF,
        model="discounting",
        metrics=["carry_total", "cs01", "breakeven"],
        pricing_options=opts,
    )
    vr = ValuationResult.from_json(out)
    results[h] = {
        "carry": vr.get_metric("carry_total"),
        "cs01": vr.get_metric("cs01"),
        "breakeven": vr.get_metric("breakeven"),
    }

print(f"{'Horizon':>8s}  {'Carry ($)':>12s}  {'CS01 ($/bp)':>12s}  {'Breakeven (bp)':>15s}")
print("-" * 55)
for h in horizons:
    r = results[h]
    print(f"{h:>8s}  {r['carry']:>12,.2f}  {r['cs01']:>12,.2f}  {r['breakeven']:>15.1f}")

 Horizon     Carry ($)   CS01 ($/bp)   Breakeven (bp)
-------------------------------------------------------
      1M      3,479.68       -385.31              9.0
      3M     10,169.33       -385.31             26.4
      6M     20,661.92       -385.31             53.6
      1Y     42,016.42       -385.31            109.0


## 3. Breakeven for yield (DV01-based)

The same framework works for any valuation parameter. To ask "how much can yields rise before I lose money?", switch the target to `ytm` — the calculator uses DV01 as the sensitivity instead of CS01.

In [5]:
targets = [
    ("z_spread", "CS01", "Spread BE"),
    ("ytm",      "DV01", "Yield BE"),
]

print(f"{'Target':>12s}  {'Sensitivity':>12s}  {'6M Carry ($)':>13s}  {'Sens ($/bp)':>12s}  {'BE (bp)':>10s}")
print("-" * 65)

for target, sens_name, label in targets:
    sens_metric = "cs01" if target == "z_spread" else "dv01"
    opts = json.dumps({
        "theta_period": "6M",
        "breakeven_config": {"target": target, "mode": "linear"}
    })
    out = price_instrument_with_metrics(
        bond_json, market_json, AS_OF,
        model="discounting",
        metrics=["carry_total", sens_metric, "breakeven"],
        pricing_options=opts,
    )
    vr = ValuationResult.from_json(out)
    carry = vr.get_metric("carry_total")
    sens = vr.get_metric(sens_metric)
    be = vr.get_metric("breakeven")
    print(f"{label:>12s}  {sens_name:>12s}  {carry:>13,.2f}  {sens:>12,.2f}  {be:>10.1f}")

      Target   Sensitivity   6M Carry ($)   Sens ($/bp)     BE (bp)
-----------------------------------------------------------------
   Spread BE          CS01      20,661.92       -385.31        53.6
    Yield BE          DV01      20,661.92       -375.70        55.0


## Takeaways

- **`breakeven`** is a first-class metric — request it alongside `carry_total` and the relevant sensitivity (`cs01`, `dv01`, `vega`, etc.)
- Configure via `pricing_options` JSON: `theta_period` sets the carry horizon, `breakeven_config` sets the target parameter and solve mode
- **Linear mode** (`-carry / sensitivity`) is the default — fast, transparent, what most trading desks use for screening
- **Iterative mode** (`"mode": "iterative"`) uses Brent root-finding with full reprice, accounting for convexity — use for large carry or precise analysis
- Supported targets: `z_spread`, `ytm`, `implied_vol`, `base_correlation`, `oas`
- Longer horizons produce larger breakevens (more carry cushion)
- Sign convention: positive = parameter can move against you; negative = carry is negative, parameter must move in your favour to break even